# Introduction

This notebook accompanies the (*link soon to be posted here*) ItsADataThing series. 

The goal of this notebook is to demonstrate a data pull procedure using IMDb's Non-Comercial Dataset, including how to pull, understand, and prepare the data for a graph database.

## Obtaining the Data

The first portion of this notebook downloads the IMDb datasets used for this project. The data is obtained directly from IMDb's publicly available dataset files and includes information about titles, people, cast and crew, and ratings.

The downloaded files will be stored in the user's local **Downloads** directory in a folder named `imdb-downloads`. The notebook does not store the IMDb datasets in the project repository.

### IMDb Data Usage Policy

IMDb provides these datasets for specific permitted uses and subject to IMDb's applicable terms and conditions. The datasets are intended for **non-commercial use**, and the data should not be redistributed or used for commercial purposes without the appropriate permission or license from IMDb. Users should review IMDb's current dataset terms before downloading or using the data.

This project downloads the data for educational purposes. The raw IMDb dataset files are **not included in this repository**. Each user downloads the data directly from IMDb when running the notebook.

The IMDb datasets are distributed as compressed `.tsv.gz` files. Because the datasets are large, downloading and extracting them requires substantial disk space.

For additional information about IMDb's current data products and licensing, see the [IMDb Bulk Data Documentation](https://developer.imdb.com/documentation/bulk-data-documentation/?utm_source=chatgpt.com).


In [1]:
import gzip
import requests
import shutil
from pathlib import Path

To download the data from IMDb's host site we will use the python requests library. The block below sets up the ouput path for the files. Currently it is set to your machines downloads folder, but this can be changed to fit your setup.

In [2]:
# Configuration for IMDb downloads
output_dir = Path.home() / "Downloads" / "imdb-downloads"
output_dir.mkdir(parents=True, exist_ok=True)

files_to_download = [
    ("name.basics", "https://datasets.imdbws.com/name.basics.tsv.gz"),
    ("title.basics", "https://datasets.imdbws.com/title.basics.tsv.gz"),
    ("title.principals", "https://datasets.imdbws.com/title.principals.tsv.gz"),
    ("title.ratings", "https://datasets.imdbws.com/title.ratings.tsv.gz"),
]

print(f"Using download folder: {output_dir}")

Using download folder: /Users/catalina/Downloads/imdb-downloads


This block downloads each IMDb dataset from the URLs defined in files_to_download and saves the extracted .tsv files


In [3]:
# Download and unpack each IMDb dataset
for file_name, url in files_to_download:
    gz_path = output_dir / f"{file_name}.tsv.gz"
    tsv_path = output_dir / f"{file_name}.tsv"

    if tsv_path.exists():
        print(f"{file_name}.tsv already exists; skipping download")
        continue

    print(f"Downloading {file_name}...")
    response = requests.get(url, stream=True, timeout=60)
    response.raise_for_status()

    with gz_path.open("wb") as f:
        shutil.copyfileobj(response.raw, f)

    with gzip.open(gz_path, "rt", encoding="utf-8") as gz_file:
        with tsv_path.open("w", encoding="utf-8") as out_file:
            shutil.copyfileobj(gz_file, out_file)

    gz_path.unlink(missing_ok=True)
    print(f"Saved {tsv_path}")

print("\nAll files downloaded and unpacked successfully.")

name.basics.tsv already exists; skipping download
title.basics.tsv already exists; skipping download
title.principals.tsv already exists; skipping download
title.ratings.tsv already exists; skipping download

All files downloaded and unpacked successfully.


## Preparing the Data

This notebook uses the **Pandas** library to load, clean, and transform the IMDb data. 

In [4]:
import pandas as pd

Because the seedpair list is so small, I've included the resolved IMDb IDs in the seed_pairs file. Which means we only need to resolve connecting title IDs.

In [5]:
seed_pairs = pd.read_csv("./seed_pairs.csv")
names_basics = pd.read_csv(output_dir / "name.basics.tsv", sep="\t", low_memory=False)
title_basics = pd.read_csv(output_dir / "title.basics.tsv", sep="\t", low_memory=False)
title_principals = pd.read_csv(output_dir / "title.principals.tsv", sep="\t", low_memory=False)
title_ratings = pd.read_csv(output_dir / "title.ratings.tsv", sep="\t", low_memory=False)

# Understanding The Data

Before we get into the transformation process we need to look at our data, understand the format, and decide what components are going to be useful to include in the graph. Run the below blocks to see the structure and a sample of each dataset.

In [7]:
names_basics.dtypes

nconst               str
primaryName          str
birthYear            str
deathYear            str
primaryProfession    str
knownForTitles       str
dtype: object

Note: We should probably change the type of `birthYear`, `deathYear` to integers inside the graph.

In [ ]:
names_basics.sample(5)

In [ ]:
title_basics.dtypes

Note: We should probably change the type of `startYear`, `endYear` and `runtimeMinutes` to integers inside the graph.

In [ ]:
title_basics.sample(5)

In [ ]:
title_principals.dtypes

In [ ]:
title_principals.sample(5)

In [ ]:
title_ratings.dtypes

In [ ]:
title_ratings.sample(5)

Using these datasets, we want to construct a graph that answers the question *"can we predict which combinations of actors and directors actually tend to produce well-reviewed films for proposed cast/director lineup for projects?"* We've already created a data model, so now we need to decide which columns are going to be useful in answering this question. For example I plan to keep the id columns in each data set. This includes  `nconst` and `tconst`. 

The genre column is inside the `title_basics` dataset, so we'll need to create a unique id for each genre we pull from that column. To be consistent we can call this id `gconst`. 

`name_basics` includes a `knownForTitles` column which is an abreviated list of titles these people have worked on. We do want to grab the person->title relationship but a better dataset for that is `title_principals` which contains a `tconst` `nconst` and `category` which is the role of the person on that film.

The data inside the basics datasets looks to be better for filling in metadata/properites for all of our nodes.

![Data Model (no properties)](/Users/catalina/Documents/graph-db-bakeoff/resources/datamodel1.png)


--- 
Since we have't incldued any properties in our data model yet, we need to decide which columns could be useful properties to incldue. 

In `name_basics` has to columns `primaryName` `birthYear` `deathYear` `primaryProfession` `knownForTitles`. Since we are pulling the titles people have worked on from a different dataset and will show that in a relationship, it is redundant to include `knownForTitles` as a property as well.

In `title_basics` there is an `endYear` column, this data wouldnt be useful in our graph because we are only keeping movies and documentaries which dont have an end year like a tv series. We can also leave out `genres` as a properity sicne this is another node type.

In `ratings_basics` we can pull the `averageRating` and `numVotes` columns to include as properties of our title nodes.

Finally, in `title_principals` its not necessary to incldue `characters` in our graph because we arent trying to answer any questiosn that rely on the role someone played in a title.

![Data Model (no properties)](/Users/catalina/Documents/graph-db-bakeoff/resources/datamodel2.png)

## Step 1: Resolve all connecting titles (titles where the actor-director pairs worked together)

In [ ]:
# Only director credits
directors = title_principals[title_principals["category"] == "director"]

# Only acting credits
actors = title_principals[title_principals["category"].isin(["actor", "actress"])]

In [ ]:
#Pull only directors in the seed list
director_titles = directors.merge(seed_pairs, left_on="nconst", right_on="director_id", how="inner")
#Pull only actors in the seed lsit
actor_titles = actors.merge(seed_pairs, left_on="nconst", right_on="actor_id", how="inner")
#Pull only titles that have both a director and an actor in the seed list
connected_titles = director_titles.merge(actor_titles,on=["tconst", "director_id", "actor_id"],how="inner")["tconst"]

## Prepare Title Nodes

In [ ]:
#add metadata and only pull titleType of movie or documentary
titles = title_basics.merge(connected_titles, on="tconst", how="inner").merge(
    title_ratings, on="tconst", how="left").drop(columns="endYear").drop(columns="genres").query(
        "titleType == 'movie'" or "titleType == 'Documentary'")

## Prepare Genre Nodes

In [ ]:
#pull the genres out from the titles and give them an id
genres = (
    titles["genres"]
    .dropna()
    .str.split(",")
    .explode()
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
    .to_frame(name="genre")
)

genres["gconst"] = genres.index + 1

## Prepare title_genre Relationship

In [ ]:
#create title genre join table with title id and genre id
title_genre = (
    titles[["tconst", "genres"]]
    .dropna()
    .assign(genre=lambda df: df["genres"].str.split(","))
    .explode("genre")
    .merge(genres, on="genre")
    [["gconst", "tconst"]]
)

## Step 3: Expand one hop out (pull all other cast members and directors from titles) and prepare title_person relationship

In [ ]:

title_person = (
    title_principals
    .merge(titles, on="tconst", how="inner")
    .query("category in ['actor', 'actress', 'director']")
    [["tconst", "nconst", "category"]]
)

## Step 4: prepare person nodes

In [ ]:
#grab all the unique person ids and add metadata
persons = names_basics[names_basics["nconst"].isin(title_person["nconst"])].drop(
    columns="knownForTitles", errors="ignore")


## Export Final Dataset

export dataset to Downloads folder. This Path is directred to the Neo4j isntance import folder in the video series. Since this is unique to each user I have it default to the Downloads folder. For information on the Neo4j import folder, please refer to the video (*soon to be linked here*)

In [ ]:
from pathlib import Path

output_dir = Path.home() / "Downloads" / "dataset"
output_dir.mkdir(parents=True, exist_ok=True)

# Node CSVs
persons.to_csv(
    output_dir / "persons.csv",
    index=False
)

titles.to_csv(
    output_dir / "titles.csv",
    index=False
)

genres.to_csv(
    output_dir / "genres.csv",
    index=False
)

# Relationship CSVs
title_person.to_csv(
    output_dir / "roles.csv",
    index=False
)

title_genre.to_csv(
    output_dir / "title_genre.csv",
    index=False
)

print(f"CSV files written to: {output_dir.resolve()}")